# Entregable 3 — EDA Parte 2: Calidad · Distribuciones · Riesgos · Accionables
## Detección de variaciones en el valor unitario · Exportaciones peruanas SUNAT

**Autor:** Said Leonardo Uceda Paredes &nbsp;|&nbsp; **Programa:** Maestría en IA — UNI &nbsp;|&nbsp; **Sprint 1 · Semana 3** &nbsp;|&nbsp; Mayo 2026

---
| Sección | Contenido |
|---------|----------|
| 1 | **Calidad** — Nulos · Tipos · Rangos · Duplicados |
| 2 | **Distribuciones y relaciones** — Histogramas · Balance · Correlaciones |
| 3 | **Riesgos** — Desbalance · Leakage · Drift · Sesgo |
| 4 | **Conclusiones accionables** — Mínimo 2 decisiones concretas |

In [ ]:
import logging
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
log = logging.getLogger('eda2')

plt.rcParams.update({
    'figure.dpi'         : 120,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.titlesize'     : 10,
    'axes.labelsize'     : 9,
})

# ── Parámetros de conexión ─────────────────────────────────────────────────────
SERVIDOR     = r'DESKTOP-OGU19A7\SQLEXPRESS,56878'
BASE_DATOS   = 'DB_GEE_DW_ADUANAS'
ESQUEMA      = 'SC_ADUANA'
DRIVER       = 'ODBC+Driver+17+for+SQL+Server'
FEC_INI      = '2024-01-01'
FEC_FIN      = '2024-12-31'

# ── Columnas clave ─────────────────────────────────────────────────────────────
COL_R = 'NUM_SPN_R'
COL_C = 'ANIO_C'
COL_V = 'MTO_VALOR_UNTARIO_V'

# Umbral: mínimo 36 reg/año (= 3 reg/mes × 12 meses)
UMBRAL_ANUAL = 36

log.info('Notebook EDA Parte 2 iniciado — %s', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

---
## Sección 1 — Calidad de datos
### 1.1 Carga desde SP `EDA_BASE`

In [ ]:
url = (
    f'mssql+pyodbc://{SERVIDOR}/{BASE_DATOS}'
    f'?driver={DRIVER}&Trusted_Connection=yes'
)

try:
    motor     = create_engine(url, echo=False, poolclass=NullPool)
    sentencia = text(
        f"EXEC [{BASE_DATOS}].[{ESQUEMA}].[SP_VALORES_UNITARIOS] "
        f"@ACCION='EDA_BASE', "
        f"@FEC_INI='{FEC_INI}', "
        f"@FEC_FIN='{FEC_FIN}'"
    )
    with motor.connect() as conn:
        resultado = conn.execute(sentencia)
        df = pd.DataFrame.from_records(resultado.fetchall(), columns=list(resultado.keys()))
except Exception as exc:
    raise RuntimeError(
        f'No se pudo conectar a SQL Server [{SERVIDOR}].\nDetalle: {exc}'
    ) from exc
finally:
    motor.dispose()

if df.empty:
    raise ValueError('SP retornó 0 registros. Verificar INGESTA y rango de fechas.')

# ── Normalizar tipos ───────────────────────────────────────────────────────────
df[COL_V]       = pd.to_numeric(df[COL_V],       errors='coerce')
df[COL_C]       = pd.to_numeric(df[COL_C],       errors='coerce').astype('Int64')
df['FOB_DOLAR'] = pd.to_numeric(df['FOB_DOLAR'], errors='coerce')
df['PESO_NETO'] = pd.to_numeric(df['PESO_NETO'], errors='coerce')

log.info('Datos cargados — %d registros | %d partidas', len(df), df[COL_R].nunique())
print(f'Registros  : {len(df):,}')
print(f'Partidas   : {df[COL_R].nunique():,}')
print(f'Años       : {sorted(df[COL_C].dropna().unique().tolist())}')
print(f'Columnas   : {list(df.columns)}')

### 1.2 Nulos, tipos y rangos

In [ ]:
# ── Resumen de calidad por columna ─────────────────────────────────────────────
cols_numericas = [COL_V, 'FOB_DOLAR', 'PESO_NETO']
cols_categ     = [COL_R, 'SECTOR', 'TIPO_PRODUCTO']

filas = []
for col in df.columns:
    nulos  = df[col].isna().sum()
    dtype  = str(df[col].dtype)
    únicos = df[col].nunique()
    fila   = {
        'Columna'   : col,
        'Tipo'      : dtype,
        'Nulos'     : nulos,
        'Nulos_%'   : round(100 * nulos / len(df), 2),
        'Únicos'    : únicos,
    }
    if col in cols_numericas:
        fila['Min']    = round(df[col].min(), 3)
        fila['Max']    = round(df[col].max(), 3)
        fila['Mediana']= round(df[col].median(), 3)
    else:
        fila['Min']    = None
        fila['Max']    = None
        fila['Mediana']= None
    filas.append(fila)

calidad = pd.DataFrame(filas).set_index('Columna')

print('=' * 70)
print('  CALIDAD DE DATOS — RESUMEN POR COLUMNA')
print('=' * 70)
print(calidad.to_string())

# ── Rango válido del valor unitario ────────────────────────────────────────────
p01  = df[COL_V].quantile(0.01)
p99  = df[COL_V].quantile(0.99)
fuera_rango = ((df[COL_V] < p01) | (df[COL_V] > p99)).sum()
print(f'\n  Rango P01–P99 de {COL_V}: [{p01:.3f}, {p99:.3f}] USD/kg')
print(f'  Registros fuera de rango P01–P99: {fuera_rango:,} ({100*fuera_rango/len(df):.2f}%)')

log.info('Calidad de datos completada')

### 1.3 Duplicados

In [ ]:
# ── Duplicados exactos ─────────────────────────────────────────────────────────
dup_exactos = df.duplicated().sum()
dup_clave   = df.duplicated(subset=[COL_C, COL_R, COL_V]).sum()

print('=' * 55)
print('  DUPLICADOS')
print('=' * 55)
print(f'  Filas totales             : {len(df):>10,}')
print(f'  Duplicados exactos        : {dup_exactos:>10,} ({100*dup_exactos/len(df):.2f}%)')
print(f'  Duplicados (año+partida+VU): {dup_clave:>10,} ({100*dup_clave/len(df):.2f}%)')

if dup_exactos > 0:
    print('\n  Muestra de duplicados exactos:')
    print(df[df.duplicated(keep=False)].head(6).to_string(index=False))

log.info('Análisis de duplicados completado — %d exactos', dup_exactos)

---
## Sección 2 — Distribuciones y relaciones
### 2.1 Balance de la variable objetivo por sector

In [ ]:
# ── Balance por SECTOR ────────────────────────────────────────────────────────
balance_sector = (
    df.groupby('SECTOR')[COL_V]
    .agg(['count', 'median', 'std'])
    .rename(columns={'count': 'N', 'median': 'Mediana_VU', 'std': 'Desv_VU'})
    .round(3)
    .sort_values('N', ascending=False)
)
balance_sector['N_%'] = (100 * balance_sector['N'] / balance_sector['N'].sum()).round(2)

print('BALANCE POR SECTOR (variable objetivo: MTO_VALOR_UNTARIO_V)')
print('=' * 65)
print(balance_sector.to_string())

# Gráfico de barras horizontales
fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(balance_sector) * 0.5)))

y_pos   = range(len(balance_sector))
etqs    = [str(s)[:35] for s in balance_sector.index]

axes[0].barh(y_pos, balance_sector['N'], color='#1a56db', alpha=0.8)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(etqs, fontsize=8)
axes[0].set_xlabel('N registros')
axes[0].set_title('Registros por sector', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].barh(y_pos, balance_sector['Mediana_VU'], color='#057a55', alpha=0.8)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(etqs, fontsize=8)
axes[1].set_xlabel('Mediana VU (USD/kg)')
axes[1].set_title('Mediana valor unitario por sector', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.1f}'))

plt.suptitle('Balance de registros y valor unitario por sector',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_balance_sector.png', dpi=120, bbox_inches='tight')
plt.show()
log.info('Balance por sector guardado: fig_balance_sector.png')

### 2.2 Correlaciones entre variables numéricas

In [ ]:
# ── Correlación: FOB_DOLAR vs PESO_NETO vs MTO_VALOR_UNTARIO_V ───────────────
df_log = pd.DataFrame({
    'log_FOB'  : np.log1p(df['FOB_DOLAR'].dropna()),
    'log_PESO' : np.log1p(df['PESO_NETO'].dropna()),
    'log_VU'   : np.log1p(df[COL_V].dropna()),
}).dropna()

corr = df_log.corr().round(3)

print('MATRIZ DE CORRELACIÓN (escala log)')
print('=' * 40)
print(corr.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=axes[0], annot_kws={'size': 11})
axes[0].set_title('Correlación de Pearson (log)', fontweight='bold')

# Scatter log_FOB vs log_VU (muestra aleatoria para no saturar)
muestra = df_log.sample(min(5_000, len(df_log)), random_state=42)
axes[1].scatter(muestra['log_FOB'], muestra['log_VU'],
                alpha=0.15, s=8, color='#1a56db')
axes[1].set_xlabel('log(1 + FOB USD)')
axes[1].set_ylabel('log(1 + VU USD/kg)')
axes[1].set_title('log(FOB) vs log(VU) — muestra 5 000 pts', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_correlaciones.png', dpi=120, bbox_inches='tight')
plt.show()
log.info('Correlaciones guardadas: fig_correlaciones.png')

### 2.3 Distribución del VU con winsorización (FE básico)

In [ ]:
# ── Winsorización P5–P95 (Feature Engineering básico según clase) ─────────────
# Ajuste SOLO sobre la distribución global; en producción: fit en train, apply en val/test.
p5  = df[COL_V].quantile(0.05)
p95 = df[COL_V].quantile(0.95)
vu_wins = df[COL_V].clip(lower=p5, upper=p95)

print(f'Winsorización P5={p5:.3f} · P95={p95:.3f} USD/kg')
print(f'Registros afectados por corte inferior : {(df[COL_V] < p5).sum():,}')
print(f'Registros afectados por corte superior : {(df[COL_V] > p95).sum():,}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

datos_orig = df[COL_V].dropna()
datos_orig = datos_orig[datos_orig > 0]

# Original
axes[0].hist(datos_orig, bins=60, color='#6b7280', edgecolor='white', alpha=0.8)
axes[0].set_title('VU original\n(escala cruda)', fontweight='bold')
axes[0].set_xlabel('USD/kg')
axes[0].set_ylabel('Frecuencia')

# Log-transform
axes[1].hist(np.log1p(datos_orig), bins=60, color='#057a55', edgecolor='white', alpha=0.85)
axes[1].set_title('log(1 + VU)\n(cola larga comprimida)', fontweight='bold')
axes[1].set_xlabel('log(1 + USD/kg)')

# Winsorizado
axes[2].hist(vu_wins.dropna(), bins=60, color='#1a56db', edgecolor='white', alpha=0.85)
axes[2].axvline(p5,  color='#e02424', linestyle='--', linewidth=1.5, label=f'P5={p5:.3f}')
axes[2].axvline(p95, color='#e02424', linestyle='--', linewidth=1.5, label=f'P95={p95:.3f}')
axes[2].set_title('VU winsorizado P5–P95\n(outliers extremos recortados)', fontweight='bold')
axes[2].set_xlabel('USD/kg')
axes[2].legend(fontsize=8)

plt.suptitle('Comparación: original · log-transform · winsorizado',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_winsor_transform.png', dpi=120, bbox_inches='tight')
plt.show()
log.info('Gráfico winsorización guardado: fig_winsor_transform.png')

---
## Sección 3 — Riesgos
### 3.1 Desbalance — subpartidas excluidas (< 36 reg/año)

In [ ]:
# ── Subpartidas que cumplen el umbral vs excluidas ────────────────────────────
conteo = (
    df.groupby([COL_C, COL_R])[COL_V]
    .count()
    .reset_index(name='N')
)
aptas    = conteo[conteo['N'] >= UMBRAL_ANUAL]
excluidas= conteo[conteo['N'] <  UMBRAL_ANUAL]

total_sp  = conteo[COL_R].nunique()
n_aptas   = aptas[COL_R].nunique()
n_excl    = excluidas[COL_R].nunique()

print('=' * 65)
print('  DESBALANCE — SUBPARTIDAS')
print('=' * 65)
print(f'  Umbral                          : {UMBRAL_ANUAL} reg/año (≈ 3/mes)')
print(f'  Total subpartidas               : {total_sp}')
print(f'  Aptas para análisis (>= {UMBRAL_ANUAL})    : {n_aptas} ({100*n_aptas/total_sp:.1f}%)')
print(f'  Excluidas (<  {UMBRAL_ANUAL})              : {n_excl} ({100*n_excl/total_sp:.1f}%)')

if not excluidas.empty:
    print(f'\n  DETALLE EXCLUIDAS:')
    print(f'  {"AÑO":<8} {"N":>5}  SUBPARTIDA')
    print(f'  {"-"*8} {"-"*5}  {"-"*50}')
    for _, row in excluidas.sort_values('N').iterrows():
        print(f'  {row[COL_C]!s:<8} {row["N"]:>5}  {str(row[COL_R])[:55]}')

# Concentración: % de registros cubiertos por top 10 partidas
top10_n  = df.groupby(COL_R)[COL_V].count().nlargest(10).sum()
pct_top10= 100 * top10_n / len(df)
print(f'\n  Concentración: TOP 10 subpartidas acaparan el {pct_top10:.1f}% de los registros')

log.info('Análisis de desbalance completado')

### 3.2 Leakage · Drift · Sesgo

In [ ]:
print('=' * 65)
print('  RIESGOS — LEAKAGE / DRIFT / SESGO')
print('=' * 65)

# ── LEAKAGE ───────────────────────────────────────────────────────────────────
print('\n  [LEAKAGE]')
print('  MTO_VALOR_UNTARIO_V = FOB/PESO calculado en SQL, sin usar ES_OUTLIER.')
print('  ES_OUTLIER se asigna en Python post-carga → no hay fuga train/test.')
print('  Riesgo residual: contaminación=0.05 fija; si la tasa real difiere,')
print('  la sensibilidad del modelo varía. Controlar con validación temporal.')

# ── DRIFT ─────────────────────────────────────────────────────────────────────
print('\n  [DRIFT]')
mediana_anual = df.groupby(COL_C)[COL_V].median().round(3)
if len(mediana_anual) > 1:
    drift_max = abs(mediana_anual.pct_change().dropna()).max() * 100
    nivel = 'ALTO' if drift_max > 30 else ('MODERADO' if drift_max > 10 else 'BAJO')
    print(f'  Variación máxima de mediana entre años : {drift_max:.1f}% — {nivel}')
    print(f'  Mediana por año: {mediana_anual.to_dict()}')
else:
    print('  Un solo año disponible — drift no evaluable.')
    print('  Con FEC_DECLARACION: evaluar drift mensual dentro del año.')

# ── SESGO por sector ──────────────────────────────────────────────────────────
print('\n  [SESGO]')
skew_global = df[COL_V].skew()
print(f'  Asimetría global (skew) : {skew_global:.3f}')
if abs(skew_global) > 1:
    print('  Distribución fuertemente sesgada a la derecha.')
    print('  → Transformación log recomendada antes de entrenar modelos basados en distancia.')

skew_sector = df.groupby('SECTOR')[COL_V].skew().round(3).sort_values(ascending=False)
print('\n  Asimetría por sector:')
for sector, sk in skew_sector.items():
    alerta = ' ← ALTO' if abs(sk) > 2 else ''
    print(f'    {str(sector)[:40]:<40} skew={sk:>7.3f}{alerta}')

print('\n' + '=' * 65)
log.info('Análisis de riesgos Parte 2 completado')

---
## Sección 4 — Conclusiones accionables
### Mínimo 2 decisiones concretas para el siguiente sprint

In [ ]:
# ── Resumen cuantitativo para las conclusiones ─────────────────────────────────
n_total      = len(df)
n_validos    = df[COL_V].notna().sum()
skew_g       = round(df[COL_V].skew(), 3)
p99_p50      = round(df[COL_V].quantile(0.99) / df[COL_V].median(), 1)
pct_excl_sp  = round(100 * n_excl / total_sp, 1)
pct_top10_r  = round(pct_top10, 1)

print('=' * 70)
print('  CONCLUSIONES ACCIONABLES — EDA PARTE 2')
print('  Said Leonardo Uceda Paredes | UNI FIIS | Maestría en IA | Mayo 2026')
print('=' * 70)

print(f'''
  ACCIONABLE 1 — Aplicar transformación log antes del modelado
  ─────────────────────────────────────────────────────────────
  Evidencia : skew global = {skew_g} (>1 = cola derecha larga).
               Ratio P99/P50 = {p99_p50}x → valores extremos distorsionan
               métricas de distancia (IQR, LOF, DBSCAN).
  Acción    : entrenar todos los modelos sobre log1p(MTO_VALOR_UNTARIO_V).
               Evaluar también winsorización P5–P95 como variante B (ablación).
  Riesgo    : reversar la transformación al reportar umbrales al negocio.
  Sprint    : implementar en Feature Engineering (Semana 4).
''')

print(f'''
  ACCIONABLE 2 — Excluir subpartidas con < {UMBRAL_ANUAL} registros/año del modelo
  ─────────────────────────────────────────────────────────────
  Evidencia : {n_excl} subpartidas ({pct_excl_sp}%) no alcanzan el mínimo estadístico
               de 3 registros/mes. IQR y percentiles son inestables con N < 36.
  Acción    : filtrar antes de entrenar; reportar aparte como "sin historial
               suficiente". Cuando se disponga de FEC_DECLARACION, recalcular
               con granularidad mensual (umbral = 3 reg/mes).
  Riesgo    : subpartidas nuevas o estacionales quedan sin cobertura.
  Sprint    : agregar flag IS_EXCLUIDA en pipeline (Semana 4).
''')

print(f'''
  ACCIONABLE 3 (adicional) — Estratificar análisis por sector
  ─────────────────────────────────────────────────────────────
  Evidencia : TOP 10 subpartidas concentran {pct_top10_r}% de los registros.
               Sesgo varía significativamente por sector (ver tabla arriba).
  Acción    : calibrar contamination del Isolation Forest por sector en lugar
               de usar 5% global; usar SECTOR como variable de agrupación.
  Riesgo    : sectores con pocos registros heredan el 5% global.
  Sprint    : evaluar en experimento A/B (Semana 4–5).
''')

print('=' * 70)
log.info('Conclusiones accionables generadas — 3 accionables')